# LLMs for Software Engineering Productivity

### A hands-on, 3.5-hour workshop on vibe coding, AI coding tools, and designing systems that leverage LLMs

> **Last updated:** June 2026 · **Format:** lecture + live demos + exercises · **Level:** intermediate (you write code; you have not necessarily built LLM systems)
>
> This is a *working* document. Keep a terminal open, keep an editor open, and run things as you go. The goal is not to admire AI coding tools — it's to leave with a workflow you trust on real code, and the judgment to know when *not* to trust the machine.

---


## Who this is for & what you'll be able to do

You already write software. By the end of this session you will be able to:

1. **Drive** an agentic coding tool (Claude Code / GitHub Copilot) on a real codebase — not just autocomplete, but plan → implement → verify loops you can walk away from.
2. **Configure** your repo so the tool is good *by default* (`CLAUDE.md`, instructions files, skills, hooks, subagents).
3. **Design** software systems that use LLMs — choosing the right tier (single call → workflow → agent), engineering context, defining tools, enforcing structured output, and writing evals.
4. **Judge** AI-generated code: spot the failure modes, the security traps, and the technical-debt patterns — and put gates in place so they don't reach production.

We optimize for **what you will actually use on Monday**, not exhaustive coverage of every feature.

---


## Agenda (≈210 minutes)

| # | Module | Time | What you leave with |
|---|--------|------|---------------------|
| 0 | Setup & ground rules | 10 min | A working environment |
| 1 | The 2026 landscape | 20 min | A mental model + the model leaderboard |
| 2 | The tools & how to drive them | 35 min | Claude Code / Copilot running locally |
| — | ☕ Break | 5 min | |
| 3 | Best practices for AI-assisted coding | 50 min | A workflow you trust |
| 4 | Design principles for LLM-powered systems | 50 min | Code patterns: calls, workflows, agents, evals |
| — | ☕ Break | 5 min | |
| 5 | Integrating AI into your SDLC | 30 min | TDD, review, refactor, CI recipes |
| 6 | Risks, security & governance | 20 min | A safety checklist for your team |
| 7 | Capstone scenarios & wrap-up | 5 min | A cheat sheet |

> **Instructor note (linear demo):** every code cell in Modules 3–5 is runnable top-to-bottom. Don't skip the setup cell. The Claude API examples all reuse one helper defined in §4.0.

---


# Module 0 — Setup & ground rules (10 min)


## 0.1 What you need

- A terminal, `git`, and Python 3.10+ (`python --version`).
- **One AI coding tool installed** (we'll use Claude Code as the reference; Copilot notes are alongside).
- For the API portion (Module 4): an Anthropic API key in your environment.

```bash


# Python deps for the hands-on API sections
pip install -U anthropic pydantic


In [ ]:
# Run once to set up this notebook's Python deps
%pip install -U anthropic pydantic


# Anthropic API key (get one from the Claude Console)
export ANTHROPIC_API_KEY="sk-ant-..."


# Claude Code (terminal-native agent)
npm install -g @anthropic-ai/claude-code   # then run: claude


# GitHub Copilot: install the extension in VS Code / JetBrains and sign in
```

> 💡 If you need to run an interactive login (e.g. `claude` auth, or `gh auth login`), and you're in a managed class environment, type the command with a leading `!` in your shell so its output is visible to everyone.


## 0.2 Three ground rules for the whole session

1. **You are the senior engineer. The model is a fast, eager junior.** It types faster than you and knows more APIs than you, but it has no taste, no accountability, and no memory of *why* your system is the way it is. You delegate; you do not abdicate.
2. **If you can't verify it, don't ship it.** Every AI change needs a check it (or you) can run: a test, a build, a diff, a screenshot. "Looks done" is not a signal.
3. **Context is the scarce resource.** Not your time, not the model's intelligence — the *relevant tokens in the window*. Most best practices below are downstream of this one fact.

---


# Module 1 — The 2026 landscape (20 min)


## 1.1 The spectrum: from autocomplete to autonomous agent

AI coding assistance is not one thing. It's a spectrum of *how much you delegate*:

```
 less delegation  ────────────────────────────────────────────►  more delegation

 [1] Inline           [2] Chat              [3] Agent              [4] Async agent
 completion           ("explain this,       ("implement OAuth,     ("here's a ticket,
 (tab to accept       write this fn")        run the tests,         open a PR")
  the next line)                             fix failures")
 ─────────────       ─────────────         ─────────────          ─────────────
 You write,          You converse,          You describe,          You assign,
 it predicts.        it drafts.             it executes & loops.    it works unattended.
 Single file.        Single file/snippet.   Multi-file, tools.      Whole task, in CI/cloud.
```

The skill is knowing **which mode fits the task**, and most of this workshop is about modes [3] and [4] — where the leverage *and* the risk live.

> **"Vibe coding," precisely.** The term (coined by Andrej Karpathy in early 2025) originally meant: *describe what you want in natural language, accept what the model produces, barely read the code.* That's a legitimate mode — for prototypes, throwaways, and learning. It is **not** a production methodology. In 2026 the industry has largely split the term in two: "vibe coding" (fast, low-scrutiny, disposable) versus **spec-driven / agentic engineering** (the model still writes most of the code, but inside specs, tests, and review gates). This workshop teaches the second while being honest about the first.


## 1.2 The model leaderboard (mid-2026)

Coding ability is commonly benchmarked on **SWE-bench Verified** (resolve real GitHub issues) and the harder **SWE-bench Pro**. As of June 2026:

| Model | SWE-bench Verified* | Input / Output ($/1M tok) | Context | Notes |
|-------|--------------------:|---------------------------|---------|-------|
| **Claude Fable 5** | ~95% | $10 / $50 | 1M | Anthropic's most capable; thinking always on; for the hardest long-horizon work |
| **Claude Opus 4.8** | 88.6% | $5 / $25 | 1M | Default top-tier for coding/agentic work |
| Claude Opus 4.7 | 87.6% | $5 / $25 | 1M | Previous-gen Opus |
| GPT-5.3 Codex / GPT-5.4 | 85% / 84% | (provider) | large | Strong coding; GPT-5.5 neck-and-neck with Opus 4.8 |
| Claude Sonnet 4.6 | — | $3 / $15 | 1M | Best speed/intelligence balance for high-volume |
| Gemini 3.1 Pro | ~75% | (provider) | very large | Leads reasoning / data analysis |
| Claude Haiku 4.5 | — | $1 / $5 | 200K | Fastest/cheapest; subagents, simple tasks |

\* *Anthropic-harness numbers where published; vendor harnesses differ — see the crucial caveat below.*

**Specialization, roughly:** Opus 4.8 / GPT-5.5 lead coding; Gemini 3.1 leads reasoning & data; Grok 4.3 is cheapest with strong tool-use. Don't religiously chase the #1 — within the top tier the differences are small, and the next point dominates.


## 1.3 The single most important insight: the harness beats the model

When the same model is run through different scaffolding, scores swing by **17–21 percentage points**. Example: one model scored ~52% on a third-party standardized harness vs ~69% on the vendor's own agent harness — *same weights*. 

> **The agent tooling moves results more than swapping models does.** The context you feed, the tools you expose, the verification loop you close, the way you manage the context window — that *harness* is where your engineering leverage is. This is good news: it means **your skill at driving the tool matters more than which frontier model you pay for.** The rest of this workshop is, essentially, harness skill.


## 1.4 The mental model we'll use all day

> **Treat the agent like a brilliant intern with infinite energy, zero context, and no judgment about your codebase.**
>
> - You wouldn't hand an intern a one-line ticket and merge their PR unread. → **Specify and review.**
> - You'd point them at the existing patterns to follow. → **Reference your code.**
> - You'd give them tests so they know when they're done. → **Verification loops.**
> - You'd expect them to ask when the task is ambiguous. → **Let it interview you.**
> - You wouldn't let them touch production secrets or run destructive commands unsupervised. → **Permissions & gates.**

Hold this model. Every practice in Modules 3–6 is a corollary of it.

### ✏️ Discussion (3 min)
Think of the last non-trivial task you shipped. **Which mode** (1–4 above) would have fit it best? What would the agent have needed from you to do it well? Keep that task in mind — you'll use it in the Module 3 exercise.

---


# Module 2 — The tools & how to drive them (35 min)


## 2.1 Three interaction modes, and when to use each

Independent of vendor, you have three gears:

| Mode | Use it for | Avoid it for |
|------|-----------|--------------|
| **Inline completion** | Boilerplate, repetitive edits, the obvious next line, single-file tweaks | Anything spanning files or needing a plan |
| **Chat / ask** | "Explain this code," "what's the bug here," scoped single edits, learning a codebase | Long multi-step changes (context gets messy) |
| **Agent** | Multi-file features, "run tests and fix failures," migrations, tasks that benefit from tool use (shell, search, web) | Tiny one-line fixes (planning overhead isn't worth it) |

A practical rule: **if you could describe the diff in one sentence, use completion or a direct ask. If it needs a plan, use the agent.**


## 2.2 Claude Code — the terminal-native agent

Claude Code is an *agentic coding environment*: it reads your files, runs commands, makes changes, and loops through problems while you watch or step away. It is not a chatbot that waits — you describe the outcome, it explores, plans, and implements.

The core loop you'll run hundreds of times:

```bash
cd your-project
claude                         # start an interactive session in the repo


# then, in the session:
> read src/auth and explain how we handle sessions and login   # explore (ask)
> add Google OAuth. what files change? make a plan.            # plan
> implement the plan. write tests for the callback, run them,  # implement + verify
  fix failures.
> commit with a descriptive message and open a PR             # ship
```

Things that matter from day one:

- **Plan mode** — before editing, the agent presents a structured plan (files it'll touch, in what order). You edit the plan, *then* let it execute. Separating "decide what to do" from "do it" is the single highest-leverage habit (Module 3).
- **Permissions** — by default it asks before writes/commands. You can allowlist safe commands (`npm run lint`), use an auto-classifier mode, or sandbox it. Tune this so you're reviewing *decisions*, not clicking *approve* fifty times.
- **`/clear`** — wipe context between unrelated tasks. Cheap and important.
- **Headless mode** — `claude -p "prompt"` runs non-interactively for scripts, CI, and batch jobs (Module 5).


## 2.3 GitHub Copilot — completions, chat, and agents

Copilot spans the same spectrum inside your IDE (and on GitHub):

- **Inline completions / next-edit suggestions** — the original Copilot; great for the [1] mode.
- **Copilot Chat** — the [2] mode in the editor.
- **Agent mode** (GA on VS Code *and* JetBrains as of 2026) — the [3] mode: it plans and executes multi-step tasks, runs commands/tests, and iterates in an agentic loop. Fully supports **MCP** (Model Context Protocol) to connect external tools, databases, and browsers.
- **Copilot coding agent** — the [4] mode: assign it a GitHub issue and it works in the background and opens a PR.

The same skills transfer — give it context, a plan, and a way to verify. Copilot's persistent-instructions mechanism is `.github/copilot-instructions.md` (the analog of `CLAUDE.md`).


## 2.4 The broader field (know they exist)

- **Cursor / Windsurf** — AI-first editors (VS Code forks) with deep agent integration and their own rules files.
- **Aider** — open-source terminal agent, git-native.
- **JetBrains AI, Amazon Q, Google Gemini Code Assist** — IDE/cloud-integrated assistants.

The vendor matters less than the **practices**. Pick one, get fluent, and your skill transfers.


## 2.5 Decision guide

```
Task arrives
   │
   ├─ One obvious line/edit, single file?         → inline completion
   │
   ├─ "Explain / what's wrong / small scoped fix"? → chat / ask
   │
   ├─ Multi-file, needs a plan, runs tests/tools? → agent (interactive)
   │     └─ unfamiliar code or risky approach?    →   ...use PLAN MODE first
   │
   └─ Well-specified, parallelizable, or batch?   → async/headless agent
```


## 2.6 🛠️ Hands-on (10 min): your first real agent task

Pick a *small but real* task in a repo you have (add a test, fix a known bug, add a CLI flag). Then:

1. Start your agent in the repo.
2. **Ask first, don't command:** "Explain how X works and where Y lives." Read the answer.
3. Ask for a **plan** ("what would you change, in what order?"). Read it. Correct anything wrong *now*.
4. Let it **implement**, and explicitly tell it to **run the test/build** afterward and fix failures.
5. Review the diff. Note: how much did you have to correct? Where did it go off track?

> **Debrief prompt:** What did the tool *not* know that you had to tell it? (That gap is exactly what `CLAUDE.md` / instructions files are for — next module.)

---


# Module 3 — Best practices for AI-assisted coding (50 min)

This is the heart of the workshop. Everything here generalizes across tools. Most of it follows from one constraint.


## 3.1 Constraint #1: the context window fills fast, and performance degrades as it fills

The model's context window holds your *entire* conversation: every message, every file it reads, every command's output. A single debugging session can burn tens of thousands of tokens. And here's the part people miss:

> **LLM performance *degrades* as context fills.** When the window gets full, the model starts "forgetting" earlier instructions and making more mistakes. This is sometimes called *context rot* — and it's been measured: across many frontier models, accuracy *drops* as input grows, even well within the advertised window.

So "just use the 1M-token window" is not a strategy. The goal of context engineering (Module 4 formalizes this) is **the smallest set of high-signal tokens that gets the job done.** Practically, in a coding session, that means: scope tightly, clear often, and don't let the agent read 200 files when it needs 3.


## 3.2 The core workflow: Explore → Plan → Implement → Commit

Letting the agent jump straight to coding is how you get *code that solves the wrong problem.* Separate research from execution:

```
1. EXPLORE   (plan mode / ask)   "read src/auth and how we manage env secrets.
                                   don't change anything yet."
2. PLAN      (plan mode)          "I want to add Google OAuth. what files change?
                                   what's the session flow? write a plan."
                                   → you edit/approve the plan
3. IMPLEMENT (default mode)       "implement the plan. write tests for the callback
                                   handler. run the suite. fix failures."
4. COMMIT                         "commit with a descriptive message and open a PR."
```

When to **skip** the plan: if you could describe the diff in one sentence (fix a typo, add a log line, rename a variable), just ask directly. Planning is for *uncertainty* — unfamiliar code, multi-file changes, or when the approach isn't obvious.

> **Pro move — let it interview you for big features.** Instead of writing a giant prompt, say: *"I want to build X. Interview me — ask about implementation, UX, edge cases, and tradeoffs I haven't considered. Don't ask obvious questions. When we're done, write a complete spec to `SPEC.md`."* Then start a **fresh session** to execute the spec with clean context. The interview surfaces decisions you'd otherwise discover painfully mid-implementation.


## 3.3 Precise prompting: the before/after that changes everything

The model can infer intent but can't read your mind. Specificity is the cheapest quality lever you have.

| Strategy | ❌ Vague | ✅ Specific |
|----------|---------|------------|
| **Scope the task** | "add tests for `foo.py`" | "write a test for `foo.py` covering the logged-out edge case. avoid mocks." |
| **Point to sources** | "why is this API so weird?" | "look through `ExecutionFactory`'s git history and summarize how its API evolved" |
| **Reference patterns** | "add a calendar widget" | "look at `HotDogWidget.php` for our widget pattern. follow it to add a calendar widget that paginates by month. use only libraries already in the repo." |
| **Describe the symptom** | "fix the login bug" | "users report login fails after session timeout. check token refresh in `src/auth/`. write a failing test that reproduces it, then fix it." |

Provide rich context the easy way: reference files (`@path/to/file`), paste screenshots/errors, give doc URLs, or pipe data in (`cat error.log | claude`).

> Vague prompts have *one* good use: exploration. "What would you improve in this file?" can surface things you wouldn't have asked. Use vagueness deliberately, not by accident.


## 3.4 Persistent context: `CLAUDE.md` and instructions files

Anything you find yourself re-explaining belongs in a file the tool reads automatically every session.

- **Claude Code:** `CLAUDE.md` (project root, checked into git; also `~/.claude/CLAUDE.md` for personal, and `CLAUDE.local.md` for personal-per-project). Generate a starter with `/init`.
- **Copilot:** `.github/copilot-instructions.md`.
- **Cursor/Windsurf:** their rules files.

A good `CLAUDE.md` is **short and high-signal.** The test for every line: *"Would removing this cause the agent to make a mistake?"* If not, cut it — a bloated file gets *ignored*.

```markdown


# CLAUDE.md  (example)


## Commands
- Test:        pytest -q           (prefer single tests, not the whole suite)
- Lint/format: ruff check . && ruff format .
- Typecheck:   mypy src/


## Code style
- Python 3.11, type hints required on public functions
- Prefer pure functions; no global state
- Use `httpx` (async), not `requests`


## Workflow
- Typecheck after a series of edits, before declaring done
- Branch naming: feat/<ticket>, fix/<ticket>


## Gotchas
- `services/billing/` talks to a live sandbox — never run its scripts without --dry-run
- Migrations live in `db/migrations/`; never edit an applied migration
```


> When the agent keeps ignoring a rule, the file is probably **too long** and the rule is getting lost. Prune ruthlessly. Treat `CLAUDE.md` like code: review it when things go wrong.

**Beyond the always-loaded file:** for knowledge that's only *sometimes* relevant, use **skills** (a `SKILL.md` the agent loads on demand) instead of bloating `CLAUDE.md`. For things that must happen *every time with zero exceptions* (e.g. run the linter after every edit), use **hooks** — deterministic scripts the harness runs at set points, which are guarantees, not the *advice* that instructions files provide.


| ✅ Include | ❌ Exclude |
|-----------|-----------|
| Commands the agent can't guess | Anything it can learn by reading code |
| Style rules that differ from defaults | Standard language conventions |
| Test instructions, preferred runners | Detailed API docs (link instead) |
| Repo etiquette (branches, PRs) | Info that changes frequently |
| Architectural decisions, non-obvious gotchas | "Write clean code" platitudes |

## 3.5 Verification loops: give the agent a check it can run

> This is the difference between a session you have to *babysit* and one you can *walk away from.*

The agent stops when work "looks done." Without a check, *you* are the verification loop — every mistake waits for you to notice. Give it something that returns pass/fail and the loop closes itself: it does the work, runs the check, reads the result, and iterates until it passes.

A "check" is anything that returns a signal the agent can read:

| Strategy | ❌ Before | ✅ After |
|----------|----------|---------|
| Give test cases | "implement an email validator" | "write `validate_email`. cases: `a@b.com`→true, `invalid`→false, `a@.com`→false. run the tests after." |
| Verify UI visually | "make the dashboard look better" | "[screenshot] implement this design. screenshot the result, compare, list differences, fix them." |
| Fix root cause | "the build is failing" | "build fails with [error]. fix it, verify the build passes. fix the root cause — don't suppress the error." |

How hard the check *gates* the stop, from light to strict:

1. **In the prompt** — "run the check and iterate" in the same message. Works today, any tool.
2. **As a goal condition** — a separate evaluator re-checks after every turn; the agent keeps going until it holds.
3. **As a deterministic hook** — a Stop hook runs your script and blocks the turn from ending until it passes.
4. **A second opinion** — a fresh-context reviewer subagent tries to *refute* the result, so the worker isn't grading itself.

And demand **evidence**, not assertions: "show me the test output / the command you ran and what it returned." Reviewing evidence is faster than re-verifying yourself, and it works for sessions you didn't watch.


## 3.6 Manage the session: course-correct, clear, compact

Conversations are persistent *and reversible* — use that.

- **Course-correct early.** The best results come from tight feedback loops. Hit `Esc` to stop mid-action (context preserved) and redirect. Don't wait for it to finish a wrong approach.
- **The two-correction rule.** If you've corrected the same issue twice, the context is now polluted with failed attempts. **`/clear` and restart** with a better prompt that incorporates what you learned. A clean session with a sharp prompt beats a long session full of dead ends — almost every time.
- **`/clear` between unrelated tasks.** The "kitchen sink session" — task A, then an unrelated question, then back to A — leaves the window full of irrelevant tokens. Reset.
- **Compact deliberately.** When context gets long, summarize what matters ("focus on the API changes") rather than letting it auto-truncate at random.
- **Rewind / checkpoints.** You can restore prior conversation/code state. This frees you to *try risky things* — if it doesn't work, rewind. (Note: checkpoints track agent changes, not external processes — not a git substitute.)


## 3.7 Subagents & adversarial review

Because context is the constraint, **subagents** (sub-tasks that run in their *own* context window and report back a summary) are one of your most powerful tools.

- **Investigation:** "use subagents to investigate how token refresh works and whether we already have OAuth utilities to reuse." The subagent reads many files; only the *summary* lands in your main context.
- **Adversarial review:** before calling a task done, have a fresh-context reviewer look *only* at the diff and your criteria — it isn't biased by the reasoning that produced the code. Built-in `/code-review` does this for bugs; or write your own:

  > "Use a subagent to review the rate-limiter diff against `PLAN.md`. Check every requirement is implemented, listed edge cases have tests, and nothing out of scope changed. Report gaps, not style preferences."

> ⚠️ A reviewer told to *find gaps* will always find some — that's what it was asked to do. Chasing every finding leads to over-engineering. Tell it to flag only what affects **correctness or the stated requirements**, and treat the rest as optional.

A related quality pattern: the **Writer/Reviewer split** across two sessions. One session implements; a *second, fresh* session reviews. The fresh context isn't attached to the code it just wrote, so it reviews more honestly. (Same idea works for tests: one writes tests, another writes code to pass them.)


## 3.8 Common failure patterns (and the fix)

| Pattern | What it looks like | Fix |
|---------|-------------------|-----|
| **Kitchen-sink session** | Unrelated tasks pile up in one context | `/clear` between tasks |
| **Correction spiral** | You correct, it's still wrong, you correct again | After 2 tries, `/clear` + better prompt |
| **Over-stuffed `CLAUDE.md`** | Rules get ignored | Prune; convert must-haves to hooks |
| **Trust-then-verify gap** | Plausible code that misses edge cases | Always provide a check; no check → don't ship |
| **Infinite exploration** | "Investigate X" → reads 200 files, context full | Scope narrowly, or delegate to a subagent |


## 3.9 ✏️ Exercises (15 min)

Use the task you picked in Module 1.

1. **Write a `CLAUDE.md` / instructions file** for one of your repos. Keep it under ~30 lines. Apply the "would removing this cause a mistake?" test to every line.
2. **Run the same task two ways** and compare:
   - (a) one vague prompt, straight to implementation;
   - (b) Explore → Plan (correct the plan) → Implement with an explicit "run the tests and fix failures" instruction.
   - How different are the diffs? How much did you intervene in each?
3. **Force a verification loop.** Give the agent a failing test (write it yourself, or have the agent write it first) and tell it not to stop until it's green. Watch it close the loop on its own.
4. **Adversarial review.** Take the diff from (2b) and ask a fresh session to review it against your intent. Did it catch anything you missed?

---


# Module 4 — Design principles for LLM-powered systems (50 min)

Now we switch hats: from *using* AI tools to *building software that uses LLMs*. Same era, different discipline. These are the principles for putting an LLM *inside* your product.

> Code in this module uses the Anthropic Python SDK and current models (mid-2026). The patterns — tiers, context engineering, tools, structured output, evals — are provider-agnostic; the syntax is Claude-specific.


## 4.0 Setup: one reusable helper


In [ ]:
# === Module 4 setup — run this first ===
import anthropic

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment

MODEL = "claude-opus-4-8"        # top-tier default for quality work
FAST  = "claude-haiku-4-5"       # cheap/fast for simple sub-tasks & classification

def call(prompt, system=None, model=MODEL, effort="medium", max_tokens=4000):
    """Single-turn helper used throughout this module.

    - adaptive thinking: the model decides how much to reason per request
    - effort: 'low' | 'medium' | 'high' | 'max' — the cost/quality dial
    """
    kwargs = dict(
        model=model,
        max_tokens=max_tokens,
        thinking={"type": "adaptive"},
        output_config={"effort": effort},
        messages=[{"role": "user", "content": prompt}],
    )
    if system:
        kwargs["system"] = system
    resp = client.messages.create(**kwargs)
    return "".join(b.text for b in resp.content if b.type == "text")

print(call("In one sentence, what is an LLM agent?"))


> **Model choice in code:** default to `claude-opus-4-8`. Drop to `claude-sonnet-4-6` for high-volume production, `claude-haiku-4-5` for simple/speed-critical work, and reach for `claude-fable-5` only for the hardest long-horizon tasks (it's pricier). Control reasoning depth with `effort`, not by switching models. For long outputs (`max_tokens` ≳ 16K), **stream** the response.


## 4.1 The golden rule: build the simplest thing that works

The biggest design mistake is reaching for an "agent" when a single call would do. Match the tier to the task:

```
Tier 1: Single LLM call      → classification, summarization, extraction, Q&A
Tier 2: Workflow             → multi-step, but YOU control the code path
Tier 3: Agent                → the MODEL decides its own path & tool use
```

The vocabulary matters:

- **Workflows** orchestrate LLMs and tools through **predefined code paths.** Predictable, testable, cheap.
- **Agents** let the LLM **dynamically direct its own process** and tool usage. Flexible, powerful, harder to predict and bound.

> **Before you build an agent, check all four:**
> 1. **Complexity** — is the task genuinely multi-step and hard to specify up front?
> 2. **Value** — does the outcome justify higher cost and latency?
> 3. **Viability** — is the model actually good at this task type?
> 4. **Cost of error** — can mistakes be caught and recovered (tests, review, rollback)?
>
> If any answer is "no," **drop to a simpler tier.** Most production LLM features are Tier 1 or Tier 2 — and they're more reliable for it.

Three design principles from Anthropic's *Building Effective Agents* run through everything below:

1. **Maintain simplicity.** Don't add abstraction you don't need. Frameworks help you start; reduce their layers as you go to production.
2. **Prioritize transparency.** Show the planning/steps explicitly — you can't debug a black box.
3. **Craft the agent–computer interface (ACI) carefully.** Tool docs and design deserve as much care as your user-facing API.


## 4.2 The augmented LLM — the building block

Every LLM system is built from one primitive: a model augmented with **retrieval**, **tools**, and **memory.**

```
                ┌──────────────────────┐
   query ─────► │   LLM                │ ─────► response
                │   • retrieval (RAG)  │
                │   • tools (actions)  │
                │   • memory (state)   │
                └──────────────────────┘
```

Tier-2 workflows and Tier-3 agents are just *compositions* of this block. Get the block right — clear retrieval, well-designed tools, deliberate memory — and the rest composes cleanly.


## 4.3 Context engineering — the core discipline

> **Context engineering:** selecting, structuring, and maintaining the information the model uses to reason — finding *the smallest set of high-signal tokens that maximize the chance of the outcome you want.* The model has a finite **attention budget**; spend it well.

Concrete techniques:

**System prompt — strike the right altitude.** Not brittle hardcoded logic, not vague hand-waving. Organize with clear sections (XML tags or markdown headers). Start minimal on your best model, then add instructions *only* in response to observed failure modes.


In [ ]:
SYSTEM = """You are a support triage assistant for an e-commerce API.

<task>
Classify each ticket and extract the order ID if present.
</task>

<rules>
- Categories: billing, shipping, technical, account, other
- If urgency is unclear, default to "normal"
- Never invent an order ID; use null if absent
</rules>
"""


**Few-shot — curate, don't enumerate.** A few diverse, canonical examples beat an exhaustive list of edge cases. Examples are "pictures worth a thousand words" — show the behavior you want.

**Retrieval — just-in-time, not everything-up-front.** Don't pre-load every doc. Keep lightweight identifiers (file paths, IDs, queries) and let the system *fetch* what it needs at runtime. This mirrors how humans work and avoids drowning the window in irrelevant text. (Claude Code does exactly this: it loads `CLAUDE.md` up front but uses `glob`/`grep` to discover files on demand.) A **hybrid** is often best: a little essential context up front for speed, plus the ability to explore.

**For long-horizon tasks, three tools:**

- **Compaction** — when nearing the limit, summarize the context and continue from the summary. Preserve decisions and open threads; discard redundant tool output.
- **Structured note-taking** — have the system write notes to a file *outside* the window (a `NOTES.md`/progress log) and read them back later. Persistent memory without paying for it in every turn.
- **Sub-agents** — spin up clean-context workers for focused sub-tasks; they return a tight 1–2k-token summary, not their whole transcript.

> **Rule of thumb:** as models improve, *reduce* prescriptive engineering and grant more autonomy — but only as far as your evals stay green. Start tight, loosen with evidence.


## 4.4 Workflow patterns (Tier 2)

Five composable patterns cover most workflow needs. Code sketches use the `call()` helper from §4.0.

### (a) Prompt chaining — decompose into sequential steps
Each step's output feeds the next. Add a gate between steps to fail fast.


In [ ]:
def write_blog(topic):
    outline = call(f"Write a tight 5-bullet outline for a blog post on: {topic}")
    # gate: cheap programmatic check before spending more tokens
    if outline.count("\n") < 3:
        raise ValueError("Outline too thin; revise the prompt.")
    draft = call(f"Write the post from this outline:\n{outline}")
    polished = call(f"Tighten this for clarity and remove fluff:\n{draft}")
    return polished


### (b) Routing — classify, then send to a specialist path
Cheaper/faster handling per category; each path can use a different model or prompt.


In [ ]:
def route(ticket):
    category = call(
        f"Classify into exactly one of [billing, technical, other]. "
        f"Reply with only the word.\n\nTicket: {ticket}",
        model=FAST, effort="low", max_tokens=10,
    ).strip().lower()

    if category == "billing":
        return call(f"As a billing specialist, resolve:\n{ticket}", effort="high")
    if category == "technical":
        return call(f"As an SRE, debug:\n{ticket}", effort="high")
    return call(f"Handle this general request:\n{ticket}")


### (c) Parallelization — fan out, then aggregate
Run independent subtasks concurrently (sectioning), or get multiple votes (voting). Great for "review this code for {security, performance, style}" in parallel.

### (d) Orchestrator–workers — a lead LLM decomposes dynamically
When you can't predict the subtasks up front (e.g. "make this change across however many files need it"), an orchestrator breaks the work down and dispatches workers. (This is the Tier-2/Tier-3 boundary.)

### (e) Evaluator–optimizer — generate, critique, refine in a loop
One LLM produces, another evaluates against criteria, the first revises. Use when you have clear quality criteria and iteration measurably helps (e.g. literary translation, complex search).


In [ ]:
def refine(task, rounds=3):
    answer = call(f"Complete this task:\n{task}")
    for _ in range(rounds):
        critique = call(
            f"Critique this answer against the task. If it fully meets the bar, "
            f"reply exactly 'APPROVED'. Else list concrete fixes.\n\n"
            f"TASK:\n{task}\n\nANSWER:\n{answer}"
        )
        if "APPROVED" in critique:
            break
        answer = call(f"Revise using this critique.\nANSWER:\n{answer}\nCRITIQUE:\n{critique}")
    return answer


## 4.5 Tools & the agent–computer interface (ACI)

Tools are how the model *acts*. Tool design is an engineering discipline of its own.

**Principles for good tools:**
- **One clear purpose each**, minimal overlap. If *you* can't say which tool applies, neither can the model.
- **Descriptive, unambiguous parameters.** The description is the contract — be **prescriptive about *when* to call it**, not just what it does ("Call this when the user asks about current prices or recent events"). Recent models reach for tools more conservatively, so trigger conditions in the description measurably raise the should-call rate.
- **Token-efficient results.** Return what's useful, compact. Don't dump 10k tokens of JSON the model will mostly ignore.
- **A few thoughtful tools beat a bloated set.** Start with the high-impact workflows; scale up from there.

The SDK's **tool runner** handles the agentic loop for you — define typed functions, it calls the API, executes your functions, and loops until done:


In [ ]:
from anthropic import beta_tool
import subprocess

@beta_tool
def run_tests(path: str = "tests/") -> str:
    """Run the pytest suite and return the output.

    Args:
        path: file or directory to test. Defaults to the whole suite.
    """
    r = subprocess.run(["pytest", path, "-q"], capture_output=True, text=True)
    return (r.stdout + r.stderr)[-4000:]   # keep results token-efficient

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=8000,
    tools=[run_tests],
    messages=[{"role": "user", "content": "Run the tests and summarize any failures."}],
)
for message in runner:        # the loop runs automatically until the model is done
    for block in message.content:
        if block.type == "text":
            print(block.text)


When you need fine-grained control — **human-in-the-loop approval**, custom logging, conditional execution — write the loop manually instead:

```python
tools = [ run_tests.to_dict() ] if False else []   # your JSON tool defs here
messages = [{"role": "user", "content": "Fix the failing tests."}]

while True:
    resp = client.messages.create(model=MODEL, max_tokens=8000, tools=tools, messages=messages)
    if resp.stop_reason == "end_turn":
        break
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for block in resp.content:
        if block.type == "tool_use":
            # 🚦 GATE destructive/irreversible tools here before executing:
            #    if block.name in DESTRUCTIVE and not approved(block): ...
            output = execute_tool(block.name, block.input)   # your dispatcher
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
    messages.append({"role": "user", "content": results})
```

> **Design tip — bash vs. dedicated tools.** A `bash` tool gives the model maximum reach but gives *your harness* only an opaque command string. Promote an action to a **dedicated tool** when you need to **gate** it (hard-to-reverse: deletes, sends, payments), **enforce invariants** (reject edits to a file changed since last read), **render** it specially, or **parallelize** safely. Rule of thumb: start with bash for breadth; promote to a typed tool when you need to gate, audit, render, or schedule the action.


## 4.6 Structured outputs — stop parsing prose

For anything programmatic, constrain the output to a schema. With Pydantic, you get validated objects directly:


In [ ]:
from pydantic import BaseModel
from typing import Optional

class Ticket(BaseModel):
    category: str            # billing | shipping | technical | account | other
    urgency: str             # low | normal | high
    order_id: Optional[str]  # null if absent
    summary: str

resp = client.messages.parse(
    model=MODEL,
    max_tokens=1000,
    system="Classify the support ticket. Never invent an order ID.",
    messages=[{"role": "user", "content":
        "My order #A1043 still hasn't shipped after two weeks and I'm furious."}],
    output_format=Ticket,
)

t = resp.parsed_output            # a validated Ticket instance
print(t.category, t.urgency, t.order_id)   # shipping high A1043


This eliminates a whole class of brittle string-parsing bugs and makes the LLM a dependable component in a larger program. (Use `strict: true` tool schemas for the same guarantee on tool inputs.)


## 4.7 RAG in one slide

When the model needs knowledge it doesn't have (your docs, your data), **retrieval-augmented generation**: fetch relevant chunks, put them in context, answer *grounded in them.*


In [ ]:
def rag_answer(question, retrieve):
    chunks = retrieve(question, k=5)              # your vector/keyword search
    context = "\n\n---\n\n".join(chunks)
    return call(
        f"Answer using ONLY the context. If it's not in the context, say so.\n\n"
        f"<context>\n{context}\n</context>\n\nQuestion: {question}",
        effort="high",
    )


Two design choices that matter:
- **Just-in-time vs. up-front** (see §4.3): retrieve at query time; don't stuff everything in.
- **Ground and cite.** Tell the model to answer *only* from the context and to admit when the answer isn't there. Hallucination usually means retrieval failed — fix retrieval before blaming the model.

> 💰 **Prompt caching** pays off hard for RAG and agents: a large, stable prefix (system prompt + retrieved corpus) can be cached so repeat requests cost ~0.1× for the cached portion. Keep the *stable* content first and the *variable* content (the user's question) last — any byte change in the prefix invalidates the cache.


## 4.8 Evals — you cannot improve what you don't measure

> If you remember one thing from this module: **models are strong; reliability comes from architecture + guardrails + evals.** A demo that works once is not a system.

For agentic systems, evaluate the **whole trajectory**, not just the final string: tool-choice correctness, argument validity, step count, cost/latency, and policy compliance. Use deterministic tool mocks in CI; reserve LLM-judge scoring for things you can't check programmatically — and always with a rubric.

**LLM-as-judge**, the workhorse eval, with structured output:


In [ ]:
from pydantic import BaseModel

class Judgment(BaseModel):
    score: int       # 1-5
    passed: bool
    reasoning: str

def judge(task, output, rubric):
    resp = client.messages.parse(
        model=MODEL,
        max_tokens=1500,
        system=("You are a strict evaluator. Score 1-5 against the rubric. "
                "passed=true only if score >= 4."),
        messages=[{"role": "user", "content":
            f"RUBRIC:\n{rubric}\n\nTASK:\n{task}\n\nOUTPUT:\n{output}"}],
        output_format=Judgment,
    )
    return resp.parsed_output

# Run it over a dataset and track pass-rate as your release gate:
def eval_suite(cases, system_under_test, rubric):
    results = [judge(c["task"], system_under_test(c["task"]), rubric) for c in cases]
    pass_rate = sum(r.passed for r in results) / len(results)
    print(f"pass rate: {pass_rate:.0%}")
    return results


**Ship evals in CI.** Treat your eval suite like a test suite: a pass-rate threshold becomes a gate on prompt/model changes. The first time a "harmless" prompt tweak drops your pass rate from 92% to 71%, you'll understand why this matters.


## 4.9 Guardrails — defense in depth

Production LLM systems wrap the model in layers:

- **Input guardrails** — validate/sanitize before the model (prompt-injection screening, PII detection).
- **Output guardrails** — schema enforcement (structured outputs), content filtering, and an output evaluation/feedback loop that scores quality and **logs failures**.
- **Action guardrails** — confirmation gates on destructive/irreversible tools; least-privilege on what the model can touch.
- **Determinism where it counts** — strict tool contracts, deterministic state transitions, and **trace-level observability** so you can replay what happened.

> Counterintuitively, the *most capable* guardrail model isn't always the right one — dedicated, cheaper safety classifiers can outperform a frontier model on false-positive rate. Match the tool to the job.


## 4.10 ✏️ Exercises (choose one or two, ~15 min)

1. **Tier discipline.** Take a feature idea ("summarize PRs," "answer from our docs," "auto-label issues"). Write down which tier it needs and *why* (run it through the four agent questions). Implement the Tier-1 or Tier-2 version with `call()`.
2. **Build a router** for 3 categories of your choosing and test it on 5 inputs. Use `FAST`+`effort="low"` for the classifier.
3. **Structured extraction.** Define a Pydantic model for something in your domain and extract it from 3 messy inputs with `messages.parse`.
4. **Write a 5-case eval** with an LLM judge and a rubric for one of the above. Compute the pass rate. Then change a prompt and re-run — did it move?

---


## ☕ Break (5 min)

---


# Module 5 — Integrating AI into your SDLC (30 min)

Now we connect Modules 3 and 4 to the actual software lifecycle. These are recipes you can adopt this week.


## 5.1 Test-Driven Development with AI (the highest-trust loop)

TDD and agents are a perfect match: the test *is* the verification loop.

```
1. "Write failing tests for [behavior]. Don't implement yet — I'll review the tests."
2. (You review the tests. This is the real spec. Fix anything wrong.)
3. "Now implement until these tests pass. Don't modify the tests. Run them and iterate."
4. (Optional) fresh-session review of the implementation.
```

Why it works: you review the *specification* (tests) when it's cheap and small, and the agent has an unambiguous, machine-checkable definition of done. Watch out for the agent "fixing" a failing test by weakening it — pin the tests ("don't modify the tests") and review any test changes.


## 5.2 Debugging

```
"Users report [symptom]. It likely lives in [area]. 
 First write a failing test that reproduces it. 
 Then find and fix the ROOT CAUSE — do not suppress the symptom. 
 Show me the failing test, the fix, and the now-passing run."
```

Reproduce → fix → prove. The reproduction test also becomes a regression guard.


## 5.3 Large-scale refactoring & migrations (fan-out + headless)

For a migration across many files, don't do it in one giant context. **Fan out** with the headless mode:

```bash


# 1) Generate the work list (have the agent produce it)


#    e.g. "list every file importing the old logging module" -> files.txt


# 2) Loop one isolated invocation per file
for file in $(cat files.txt); do
  claude -p "Migrate $file from the old logger to structured logging. \
    Follow the pattern in src/log.py. Run that file's tests. Reply OK or FAIL." \
    --allowedTools "Edit,Bash(pytest *),Bash(git commit *)"
done
```

Refine the prompt on the first 2–3 files, *then* run at scale. `--allowedTools` scopes permissions — important when running unattended. Each file gets a clean context, so quality doesn't degrade across a 2,000-file migration.


## 5.4 AI code review (with a human gate)

Two complementary uses:

- **Pre-PR self-review:** "Review this diff for bugs, edge cases, and inconsistencies with our patterns. Report findings with severity + confidence."
- **Reviewer subagent / fresh session:** a context that didn't write the code reviews more honestly (§3.7).

> For review prompts, ask for **coverage, then filter** — "report every issue with a confidence + severity; a separate step will rank them." Recent models follow "only report high-severity" *too literally* and silently drop real bugs. Move the filtering to a second step.

**The human gate is non-negotiable.** AI review augments human review; it doesn't replace the human who is *accountable* for the merge. (Module 6 explains why, with data.)


## 5.5 Documentation & onboarding

- **Generate docs from code:** "write a docstring for every public function in `api/`, following Google style. Don't change behavior."
- **Onboarding:** the best use of an agent in a new codebase is *learning*. Ask it the questions you'd ask a senior engineer: "How does logging work? How do I add an endpoint? What does this `async move` block do on line 134? Why does this call `foo()` not `bar()`?" It reduces ramp-up time and load on your teammates.


## 5.6 Git, PRs, and CI

- **Commits/PRs:** "commit with a descriptive message and open a PR" — but *review the diff first.* (Use the `gh` CLI so the agent can manage issues/PRs context-efficiently.)
- **CI integration (headless):** run the agent non-interactively in pipelines —

```bash
claude -p "Review the diff in this PR for security issues. Output JSON." \
  --output-format json | your_ci_reporter
```

- **Pre-commit hooks, batch analysis, log triage** — anywhere a scriptable LLM step helps.


## 5.7 Spec-Driven Development for bigger features

For features too big for one prompt, **SDD** is the 2026 answer to vibe coding's failure modes. Every major tool now ships a flavor (GitHub Spec Kit, AWS Kiro, Claude Code's spec workflows, Cursor, etc.). The shape:

```
1. SPECIFY   — write a precise spec (the §3.2 interview produces SPEC.md):
               named files/interfaces, what's IN and OUT of scope, and an
               end-to-end verification step that proves the feature works.
2. PLAN      — agent turns the spec into an implementation plan; you approve.
3. IMPLEMENT — agent executes against the plan, verifying as it goes.
4. VERIFY    — run the end-to-end check from the spec.
```

SDD exists to kill **intent drift** — underspecified prompts where the model picks reasonable-but-wrong defaults. The developer's job shifts from typing code to **authoring precise specs** with the technical guardrails baked in (schemas, limits, constraints). Time spent making the spec precise pays off more than time spent watching the implementation.


## 5.8 ✏️ Exercise (10 min)
Pick **one** recipe and run it for real:
- Do a TDD loop on a small function (write/review tests → implement → green).
- Or write a `SPEC.md` for a feature via the interview technique, then start a fresh session to implement step 1.

---


# Module 6 — Risks, security & governance (20 min)

The uncomfortable half of the story. AI lets you produce code *faster than you can review, govern, or fully understand it.* That asymmetry is the core risk.


## 6.1 The data (mid-2026)

These are measured, not hypothetical:

- **~45% of AI-generated code samples introduce an OWASP Top-10 vulnerability** (across 100+ models tested on security-sensitive tasks). The model writes code that is *"almost, but not quite, right."*
- **Slopsquatting:** ~20% of AI-generated code references **packages that don't exist** — a predictable hallucination. Attackers pre-register those names as malicious packages, so a hallucinated `pip install` becomes a supply-chain compromise.
- **Secret leakage roughly doubles:** AI-assisted commits show a ~3.2% secret-leak rate vs ~1.5% baseline.
- **Technical debt compounds, it doesn't add.** Code duplication up, refactoring activity down; each AI commit introduces small inconsistencies (a slightly different auth pattern here, a duplicated util there) that *compound* across a codebase.
- **No one has a long-term maintenance track record.** Production vibe coding is ~18 months old. Teams are running uncontrolled experiments on production systems and *assuming* maintainability no one has demonstrated over years.


## 6.2 The common vulnerability patterns

Know these because they show up *constantly* in AI code:

| Pattern | What the model does by default |
|---------|-------------------------------|
| **Injection** (SQLi, command, XSS) | String concatenation instead of parameterized queries — unless you explicitly ask |
| **Broken auth/authz** | Hardcoded creds, missing authorization checks, endpoints public by default |
| **Information exposure** | Verbose errors, stack traces in responses, debug endpoints left on |
| **Credential logging** | Logs the *entire* request object "to help debugging" — including auth headers and tokens, in plaintext |


## 6.3 The mitigation checklist

Defense is mostly *process*, applied on top of everything in Modules 3–5:

- ✅ **Human review is mandatory** for anything reaching production. The accountable engineer reads the diff. No exceptions for "the AI wrote it."
- ✅ **SAST/secret-scanning in CI**, and a **security-reviewer subagent** on the diff: "review for injection, authz flaws, secrets in code, insecure data handling. Cite line numbers."
- ✅ **Verify every dependency the model adds exists and is the real package** (slopsquatting). Pin versions; use a lockfile; prefer an allowlist.
- ✅ **Never let the agent touch real secrets or run destructive commands unsupervised.** Use permission allowlists, sandboxing, `--dry-run` defaults, and confirmation gates on irreversible actions.
- ✅ **Parameterize queries; validate at boundaries; scrub logs.** Put these as explicit rules in `CLAUDE.md`/instructions so they're the *default*, not an afterthought.
- ✅ **Source-control discipline.** AI generates high volumes of change — keep commits small and reviewable, and protect main with required checks.
- ✅ **Tests + evals as gates** (Modules 4–5). If you can't verify it, don't ship it.

> A handy `CLAUDE.md` security stanza:
> ```markdown
> ## Security (non-negotiable)
> - Parameterized queries only — never string-concatenate SQL
> - Validate & sanitize all external input at the boundary
> - Never log request bodies, headers, tokens, or secrets
> - No secrets in code; read from env/secret manager
> - Every new endpoint needs an explicit authz check
> ```


## 6.4 Team governance (the meta-level)

- **Set policy on tiers of autonomy:** what can be merged from an async agent vs. what needs a human pair.
- **Treat instructions files / specs as shared, reviewed artifacts** — they're now part of your codebase's quality system.
- **Measure outcomes, not vibes:** track defect rates, review load, and rework on AI-assisted changes. Adjust your gates with data.
- **The goal is leverage *with* control:** ship faster *and* keep the ability to review, govern, and understand what you ship.


## 6.5 ✏️ Discussion (5 min)
Look back at a recent AI-generated change you merged. Run it past §6.2: did it parameterize queries? Validate input? Add any dependency you didn't verify? Log anything sensitive? What gate would have caught the worst case *automatically*?

---


# Module 7 — Capstone scenarios & wrap-up (5 min)


## 7.1 Real-world scenarios (pick one to take home)

1. **The flaky test.** A test passes locally, fails in CI ~10% of the time. Use the debugging recipe (§5.2): reproduce reliably first, then fix the *root cause* (race? time? order dependency?). Note how the agent tends to declare "fixed" after one clean run — make it prove stability.
2. **The framework migration.** Migrate a module from one HTTP client to another across ~30 files. Use fan-out (§5.3). Refine on 3 files, then batch. Measure: how many came back FAIL, and why?
3. **The greenfield feature.** Build a small feature end-to-end with SDD (§5.7): interview → `SPEC.md` → plan → implement → verify. Compare the experience to vibe-coding the same thing.
4. **The LLM feature.** Add an LLM-powered capability to an app (auto-summarize, classify, extract). Pick the right tier (§4.1), engineer the context (§4.3), enforce structured output (§4.6), and write a 5-case eval (§4.8) *before* you call it done.


## 7.2 The one-page cheat sheet

```
DRIVING THE TOOL
  • Explore → Plan → Implement → Commit. Skip the plan only for one-sentence diffs.
  • Be specific: name files, point to patterns, describe the symptom + fix criteria.
  • Give it a CHECK it can run. No check → don't ship.
  • Two failed corrections → /clear and restart with a sharper prompt.
  • Context is the budget. Scope tight, clear often, use subagents to investigate.
  • CLAUDE.md / instructions: short, high-signal, "would removing this cause a mistake?"

BUILDING WITH LLMs
  • Simplest tier that works: single call → workflow → agent. Justify every agent.
  • Augmented LLM = model + retrieval + tools + memory.
  • Context engineering: smallest high-signal token set; retrieve just-in-time.
  • Tools: one purpose each, prescriptive "when to call" docs, token-efficient results.
  • Structured outputs for anything programmatic. Evals in CI as a release gate.

SAFETY
  • ~45% of AI code has a vuln; verify deps (slopsquatting); secrets leak 2x.
  • Human review is mandatory. SAST + security subagent on the diff.
  • Never give unsupervised access to secrets or destructive commands.

MODELS (mid-2026)
  • Coding default: claude-opus-4-8 ($5/$25). Hardest work: claude-fable-5.
  • High volume: claude-sonnet-4-6. Cheap/fast & subagents: claude-haiku-4-5.
  • Control depth with `effort`, not model swaps. The HARNESS beats the model.
```


## 7.3 What to do this week

1. Write a `CLAUDE.md` / instructions file for your main repo.
2. Adopt the **Explore → Plan → Implement → Verify** loop on one real task.
3. Add **one verification gate** (a test, a lint, a security subagent) you didn't have before.
4. If you build with LLMs: pick a feature, choose the tier deliberately, and write **5 eval cases** before shipping.

---


## Appendix A — Quick reference: Claude API patterns used here

```python
import anthropic
client = anthropic.Anthropic()                       # reads ANTHROPIC_API_KEY


# Basic call with adaptive thinking + effort
resp = client.messages.create(
    model="claude-opus-4-8", max_tokens=4000,
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},                # low | medium | high | max
    messages=[{"role": "user", "content": "..."}],
)
text = "".join(b.text for b in resp.content if b.type == "text")


# Stream long outputs (max_tokens >= ~16k)
with client.messages.stream(model="claude-opus-4-8", max_tokens=64000,
                            messages=[{"role":"user","content":"..."}]) as s:
    for chunk in s.text_stream:
        print(chunk, end="", flush=True)
    final = s.get_final_message()


# Structured output (validated Pydantic object)
from pydantic import BaseModel
class Out(BaseModel):
    label: str
    confidence: float
r = client.messages.parse(model="claude-opus-4-8", max_tokens=500,
    messages=[{"role":"user","content":"..."}], output_format=Out)
obj = r.parsed_output


# Tool use (automatic loop)
from anthropic import beta_tool
@beta_tool
def search(query: str) -> str:
    """Search the knowledge base. Call when the answer needs current/internal data.
    Args: query: the search string."""
    ...
runner = client.beta.messages.tool_runner(
    model="claude-opus-4-8", max_tokens=8000, tools=[search],
    messages=[{"role":"user","content":"..."}])
for message in runner: ...


# Prompt caching (cache a large stable prefix; put variable content LAST)
resp = client.messages.create(
    model="claude-opus-4-8", max_tokens=2000,
    system=[{"type":"text","text": LARGE_STABLE_CONTEXT,
             "cache_control":{"type":"ephemeral"}}],
    messages=[{"role":"user","content": user_question}],
)
```

> Model IDs (mid-2026): `claude-fable-5`, `claude-opus-4-8`, `claude-sonnet-4-6`, `claude-haiku-4-5`. Use exact strings — no date suffixes on these aliases.


## Appendix B — `CLAUDE.md` starter template

```markdown


# <Project name>


## Commands
- Install:    <...>
- Test:       <...>   (prefer single tests for speed)
- Lint/format:<...>
- Typecheck:  <...>
- Run:        <...>


## Code style
- <language/version, formatting, key conventions that differ from defaults>


## Architecture (only the non-obvious)
- <where things live, important boundaries, "don't touch X" rules>


## Workflow
- <verify steps before "done"; branch/PR conventions>


## Security (non-negotiable)
- Parameterized queries only; validate input at boundaries
- Never log secrets/headers/tokens; no secrets in code
- New endpoints require an explicit authz check


## Gotchas
- <live sandboxes, applied migrations, flaky areas, required env vars>
```

---


## Sources & further reading

**AI coding tools & best practices**
- [Best practices for Claude Code](https://code.claude.com/docs/en/best-practices) — Anthropic
- [Claude Code Best Practices: 12 Patterns Agentic Engineers Use](https://levelup.gitconnected.com/claude-code-best-practices-12-patterns-agentic-engineers-use-65264e3eb919)
- [Optimizing Agentic Coding: How to Use Claude Code in 2026](https://aimultiple.com/agentic-coding)
- [Agent mode 101: All about GitHub Copilot's powerful mode](https://github.blog/ai-and-ml/github-copilot/agent-mode-101-all-about-github-copilots-powerful-mode/) — GitHub
- [GitHub Copilot Agent Mode 2026: Complete Tutorial](https://weavai.app/blog/en/2026/05/26/github-copilot-agent-mode-2026-complete-tutorial/)
- [GitHub Copilot Best Practices for Top Teams (2026)](https://www.metacto.com/blogs/github-copilot-best-practices-from-high-performing-teams)

**Designing LLM systems**
- [Building Effective AI Agents](https://www.anthropic.com/research/building-effective-agents) — Anthropic (workflows vs agents, the patterns)
- [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) — Anthropic
- [Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents) — Anthropic
- [The LLM context problem in 2026: strategies for memory, relevance, and scale](https://blog.logrocket.com/llm-context-problem-strategies-2026/) — LogRocket
- [AI Agents in 2026: Tools, Memory, Evals, and Guardrails](https://andriifurmanets.com/blogs/ai-agents-2026-practical-architecture-tools-memory-evals-guardrails)
- [Mastering LLM Guardrails: Complete 2026 Guide](https://orq.ai/blog/llm-guardrails)

**Model landscape**
- [Best AI Model for Coding (June 2026): 12 Models Ranked](https://www.morphllm.com/best-ai-model-for-coding)
- [AI Model Benchmarks Jun 2026 — LM Council](https://lmcouncil.ai/benchmarks)
- [Best AI Models for Coding in 2026: Claude, Codex & Gemini Compared](https://teamai.com/blog/ai-automation/best-ai-models-for-coding-and-agentic-workflows-2026/)

**Risks, security & methodology**
- [Vibe Coding's Security Debt: The AI-Generated CVE Surge](https://labs.cloudsecurityalliance.org/research/csa-research-note-ai-generated-code-vulnerability-surge-2026/) — Cloud Security Alliance
- [2026 Predictions: It's the Year of Technical Debt (Thanks to Vibe-Coding)](https://www.salesforceben.com/2026-predictions-its-the-year-of-technical-debt-thanks-to-vibe-coding/)
- [The Real Risk of Vibecoding](https://www.trendmicro.com/en_us/research/26/c/the-real-risk-of-vibecoding.html) — Trend Micro
- [Vibe Coding vs. Spec-Driven Development in 2026](https://intercode.com/blog/vibe-coding-vs-spec-driven-development-in-2026)
- [Spec-Driven Development (SDD): The Definitive 2026 Guide](https://thebcms.com/blog/spec-driven-development)
- [Vibe Coding in Practice: Flow, Technical Debt, and Guidelines (arXiv)](https://arxiv.org/abs/2512.11922)

*Built for IITR · LLMs for SWE Productivity · June 2026.*
